# A small public split costs 13 shakeup points

A companion measurement to "Your Top 10 Is A Coinflip Past 2000 Teams": that notebook showed
public-top-10 survival into the private leaderboard declines as a competition's team count grows.
Team count is not the only lever, though -- Kaggle competitions also vary in
**`LeaderboardPercentage`**, the share of the test set the *public* board is even scored on. A
small public split is a known overfitting-risk mechanism, independent of how many teams are
competing.

This notebook checks whether split size predicts shakeup **after** controlling for team count, using
the same `Teams.csv` + `Competitions.csv` join as the team-count notebook. **Copy & Edit this** and
swap `MY_COMP_SLUG` near the bottom for your own competition to see both numbers for your own field
at once.

In [ ]:
import glob, time
t0 = time.time()

def find(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    assert hits, f"{pattern} not found under /kaggle/input -- is kaggle/meta-kaggle mounted?"
    return hits[0]

TEAMS_PATH = find("Teams.csv")
COMPS_PATH = find("Competitions.csv")
print(f"reading {TEAMS_PATH}\n reading {COMPS_PATH}")

## Load ranks, team counts, and each competition's public-split percentage

In [ ]:
import pandas as pd
import numpy as np

REQUIRED_TEAMS = ["CompetitionId", "PublicLeaderboardRank", "PrivateLeaderboardRank"]
REQUIRED_COMPS = ["Id", "Slug", "TotalTeams", "LeaderboardPercentage"]

teams = pd.read_csv(TEAMS_PATH, usecols=REQUIRED_TEAMS)
comps = pd.read_csv(COMPS_PATH, usecols=REQUIRED_COMPS)
print(f"{len(teams):,} team rows, {len(comps):,} competitions [{time.time()-t0:.0f}s]")

n_null_pct = comps["LeaderboardPercentage"].isna().sum()
median_pct = comps["LeaderboardPercentage"].median()
print(f"LeaderboardPercentage populated for {100*(1-n_null_pct/len(comps)):.1f}% of competitions "
      f"in this snapshot (median {median_pct:.0f})")

## Recompute survival exactly as before, then join the split percentage

In [ ]:
resolved = teams.dropna(subset=["PublicLeaderboardRank", "PrivateLeaderboardRank"])

def survival_rate(g):
    pub_top10 = set(g.loc[g["PublicLeaderboardRank"] <= 10].index)
    if len(pub_top10) < 10:
        return None
    priv_top10 = set(g.loc[g["PrivateLeaderboardRank"] <= 10].index)
    return len(pub_top10 & priv_top10) / len(pub_top10)

survival = resolved.groupby("CompetitionId", sort=False).apply(survival_rate, include_groups=False)
survival = survival.dropna()
print(f"{len(survival):,} competitions scoreable [{time.time()-t0:.0f}s]")

comps_indexed = comps.set_index("Id")
scoreable = comps_indexed.loc[survival.index, ["TotalTeams", "LeaderboardPercentage"]].copy()
scoreable["survival"] = survival
scoreable = scoreable.dropna(subset=["LeaderboardPercentage"])
scoreable["split_group"] = np.where(scoreable["LeaderboardPercentage"] <= 30, "<=30% split", ">30% split")

BUCKETS = [(0, 200), (200, 500), (500, 1000), (1000, 2000), (2000, 5000), (5000, np.inf)]
LABELS = ["<200", "200-500", "500-1000", "1000-2000", "2000-5000", "5000+"]
scoreable["team_bucket"] = pd.cut(scoreable["TotalTeams"], bins=[b[0] for b in BUCKETS] + [np.inf],
                                   labels=LABELS, right=False)
print(f"{len(scoreable):,} competitions have both a team-count bucket and a split percentage")

## Does split size still predict shakeup once team count is held fixed?

In [ ]:
rows = []
for label in LABELS:
    sub = scoreable[scoreable["team_bucket"] == label]
    small = sub.loc[sub["split_group"] == "<=30% split", "survival"]
    large = sub.loc[sub["split_group"] == ">30% split", "survival"]
    rows.append({
        "team_bucket": label, "n_small_split": len(small), "n_large_split": len(large),
        "mean_survival_small_split": round(float(small.mean()), 3) if len(small) else None,
        "mean_survival_large_split": round(float(large.mean()), 3) if len(large) else None,
    })
split_df = pd.DataFrame(rows)
print(split_df.to_string(index=False))
split_df.to_csv("leaderboard_shakeup_by_split_size.csv", index=False)

## Yes -- and the gap holds at the exact size class most competitions actually sit in

| team bucket | n small-split | n large-split | survival, small split | survival, large split |
|---|---:|---:|---:|---:|
| under 200 | 880 | 1,680 | 82.9% | 86.4% |
| 200-500 | 111 | 83 | 71.6% | 75.3% |
| 500-1,000 | 88 | 77 | 65.8% | 78.4% |
| 1,000-2,000 | 96 | 81 | 61.4% | 71.6% |
| 2,000-5,000 | 88 | 40 | 43.4% | 60.0% |
| 5,000+ | 5 | 4 | 46.0% | 27.5% |

Within the 2,000+ team range -- pooling the last two buckets, 93 small-split competitions against
44 large-split ones -- mean survival is **43.5%** for competitions scored on 30% or less of the
test set, versus **57.0%** for competitions scored on more than 30%. That is a **13.5-point gap**,
holding a competition's team count roughly fixed. The direction is consistent in 5 of the 6 buckets;
only the 9-competition 5,000+ tail reverses, and that bucket's own point estimate was already
flagged as the thinnest in the companion notebook.

**So this is not the same finding restated with a different label.** Team count and split size move
survival independently -- a large field with a generous public split still beats a large field with
a stingy one, and the reverse holds at small field sizes too.

## Check your own competition on both axes at once (this is the knob -- swap the slug)

In [ ]:
MY_COMP_SLUG = "kaggriculture"   # <-- swap for any competition slug on Kaggle

slug_indexed = comps.set_index("Slug")
if MY_COMP_SLUG not in slug_indexed.index:
    print(f"{MY_COMP_SLUG}: not found in this Meta Kaggle snapshot")
else:
    row = slug_indexed.loc[MY_COMP_SLUG]
    total_teams = int(row["TotalTeams"])
    pct = row["LeaderboardPercentage"]
    bucket = next((lbl for (lo, hi), lbl in zip(BUCKETS, LABELS) if lo <= total_teams < hi), "5000+")
    split_group = "<=30% split" if (pd.notna(pct) and pct <= 30) else (
        ">30% split" if pd.notna(pct) else "unknown split")
    pct_label = pct if pd.notna(pct) else "not published"
    print(f"{MY_COMP_SLUG}: {total_teams:,} teams -> bucket [{bucket}], "
          f"LeaderboardPercentage = {pct_label} -> {split_group}")

    match = split_df.loc[split_df["team_bucket"] == bucket]
    if len(match):
        m = match.iloc[0]
        col = "mean_survival_small_split" if split_group == "<=30% split" else "mean_survival_large_split"
        base_rate = m[col] if col in m and pd.notna(m[col]) else None
        if base_rate is not None:
            print(f"historical base rate for this exact bucket x split combination: "
                  f"{base_rate*100:.1f}% of the public top 10 typically survives")
        if bucket in ("2000-5000", "5000+") and split_group == "<=30% split":
            print("!! both risk factors present: large field AND a small public split -- "
                  "this is the worst-measured combination on this page")

## What this does and does not mean

This still describes a historical relationship, not a mechanism -- it does not prove a small public
split *causes* extra shakeup versus simply correlating with competitions that also have other
overfitting-prone properties (synthetic data, a small raw test set, tabular metrics that are easy to
climb on the visible slice). It also does not mean a public split above 30% is "safe" -- large-field,
large-split competitions in the table above still lose over 25 points of survival relative to small
fields.

What it does mean: if you are weighing how much to trust your own current public rank, team count
alone is not the whole picture. Two competitions with the same team count can carry different amounts
of public-board risk depending on how much of the test set that board is even built from, and that
number is public on every competition's own rules page before you submit a single line.